# Coref Result Comparison

Compare original dataset text, local NLP coref output, and LLM coref output by `domain` and `doc_id`.

Expected inputs:
- `data/coref/*_coref.json` from `python -m src.nlp.coref --method nlp`
- `data/coref_llm/*_llm_coref.jsonl` from `python -m src.nlp.coref --method llm ...`


In [8]:
from pathlib import Path
import json
import sys
import difflib

import pandas as pd
from IPython.display import HTML, display


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "data" / "easdrl").exists() and (path / "src").exists():
            return path
    raise RuntimeError("Could not find project root containing data/easdrl and src")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.nlp.coref import DOMAINS, load_labeled_dataset, sample_to_text

DATA_DIR = ROOT / "data" / "easdrl"
NLP_DIR = ROOT / "data" / "coref"
LLM_DIR = ROOT / "data" / "coref_llm"

ROOT

WindowsPath('C:/anu/llm-action-extraction')

In [9]:
def load_original_records():
    records = []
    for domain in DOMAINS:
        dataset = load_labeled_dataset(domain, data_dir=str(DATA_DIR))
        for doc_id, sample in enumerate(dataset):
            records.append({
                "domain": domain,
                "doc_id": doc_id,
                "source_file": str(DATA_DIR / DOMAINS[domain]),
                "original_text": sample_to_text(sample),
            })
    return pd.DataFrame(records)


def load_nlp_records():
    records = []
    for path in sorted(NLP_DIR.glob("*_coref.json")):
        if path.name == "all_domains_coref.json":
            continue
        data = json.loads(path.read_text(encoding="utf-8"))
        for item in data:
            coref = item.get("coref", {})
            records.append({
                "domain": item["domain"],
                "doc_id": item["doc_id"],
                "nlp_resolved_text": coref.get("resolved_text"),
                "nlp_replacements": coref.get("replacements", []),
                "nlp_replacement_count": len(coref.get("replacements", [])),
            })
    return pd.DataFrame(records)


def load_llm_records():
    records = []
    for path in sorted(LLM_DIR.glob("*_llm_coref.jsonl")):
        if path.name == "all_domains_llm_coref.jsonl":
            continue
        with path.open("r", encoding="utf-8") as file:
            for line in file:
                if not line.strip():
                    continue
                item = json.loads(line)
                records.append({
                    "domain": item["domain"],
                    "doc_id": item["doc_id"],
                    "llm_resolved_text": item.get("resolved_text"),
                    "llm_model": item.get("model"),
                    "llm_response_time": item.get("response_time"),
                })
    return pd.DataFrame(records)


original_df = load_original_records()
nlp_df = load_nlp_records()
llm_df = load_llm_records()

compare_df = original_df.merge(nlp_df, on=["domain", "doc_id"], how="left")
compare_df = compare_df.merge(llm_df, on=["domain", "doc_id"], how="left")
compare_df["has_nlp"] = compare_df["nlp_resolved_text"].notna()
compare_df["has_llm"] = compare_df["llm_resolved_text"].notna()

summary = compare_df.groupby("domain").agg(
    docs=("doc_id", "count"),
    nlp_docs=("has_nlp", "sum"),
    llm_docs=("has_llm", "sum"),
    nlp_replacements=("nlp_replacement_count", "sum"),
).reset_index()
summary

,domain,docs,nlp_docs,llm_docs,nlp_replacements
0,cooking,116,116,1,153
1,wikihow,150,150,1,479
2,win2k,154,154,1,1


In [10]:
def list_docs(domain=None, only_with_llm=False, limit=30):
    df = compare_df.copy()
    if domain:
        df = df[df["domain"] == domain]
    if only_with_llm:
        df = df[df["has_llm"]]
    cols = ["domain", "doc_id", "has_nlp", "has_llm", "nlp_replacement_count", "llm_model"]
    return df[cols].head(limit)


list_docs(only_with_llm=True, limit=20)

,domain,doc_id,has_nlp,has_llm,nlp_replacement_count,llm_model
0,cooking,0,True,True,2,gpt-4o-mini
116,wikihow,0,True,True,6,gpt-4o-mini
266,win2k,0,True,True,0,gpt-4o-mini


In [11]:
def get_row(domain, doc_id):
    rows = compare_df[(compare_df["domain"] == domain) & (compare_df["doc_id"] == doc_id)]
    if rows.empty:
        raise ValueError(f"No row for domain={domain!r}, doc_id={doc_id}")
    return rows.iloc[0]


def show_texts(domain, doc_id):
    row = get_row(domain, doc_id)
    sections = [
        ("Original", row["original_text"]),
        ("NLP Coref", row.get("nlp_resolved_text") or "[missing data/coref result]"),
        ("LLM Coref", row.get("llm_resolved_text") or "[missing data/coref_llm result]"),
    ]
    html = ["<style>.coref-box{white-space:pre-wrap;border:1px solid #ddd;padding:10px;margin:8px 0}.coref-title{font-weight:700;margin-top:12px}</style>"]
    html.append(f"<h3>{domain} / doc_id={doc_id}</h3>")
    for title, text in sections:
        html.append(f"<div class='coref-title'>{title}</div><div class='coref-box'>{text}</div>")
    display(HTML("\n".join(html)))
    replacements = row.get("nlp_replacements")
    if isinstance(replacements, list) and replacements:
        display(pd.DataFrame(replacements))


show_texts("cooking", 0)

,token_index,pronoun,replacement,antecedent,antecedent_indices,source
0,191,them,soft vegetables,soft vegetables,"[179, 180]",spacy_noun_chunk
1,196,they,soft vegetables,soft vegetables,"[179, 180]",resolved_pronoun


In [12]:
def html_diff(left_text, right_text, left_label="Original", right_label="Resolved"):
    left_words = str(left_text).split()
    right_words = str(right_text).split()
    table = difflib.HtmlDiff(wrapcolumn=90).make_table(
        left_words,
        right_words,
        fromdesc=left_label,
        todesc=right_label,
        context=True,
        numlines=3,
    )
    display(HTML("<style>table.diff{font-family:monospace;font-size:12px} .diff_add{background:#d6f5d6}.diff_chg{background:#fff4b8}.diff_sub{background:#ffd6d6}</style>" + table))


def diff_doc(domain, doc_id, target="llm"):
    row = get_row(domain, doc_id)
    if target == "llm":
        resolved = row.get("llm_resolved_text")
        label = "LLM Coref"
    elif target == "nlp":
        resolved = row.get("nlp_resolved_text")
        label = "NLP Coref"
    else:
        raise ValueError("target must be 'llm' or 'nlp'")
    if not isinstance(resolved, str) or not resolved:
        raise ValueError(f"Missing {target} result for {domain}/{doc_id}")
    html_diff(row["original_text"], resolved, "Original", label)


diff_doc("wikihow", 0, target="llm")

## Common Commands

Generate local NLP results:

```powershell
C:\\Users\\Apexmod\\miniforge3\\envs\\llm\\python.exe -m src.nlp.coref --method nlp
```

Generate LLM results for a small sample:

```powershell
C:\\Users\\Apexmod\\miniforge3\\envs\\llm\\python.exe -m src.nlp.coref --method llm -m gpt-4.1-mini --limit 2
```
